# Face Recognition Project
## Using Pre-trained CNN Models

This notebook demonstrates two approaches to face recognition:
1. **Easy Approach**: Using `face_recognition` library (dlib's ResNet)
2. **Deep Learning Approach**: Using VGG-Face pre-trained model

### Dataset Recommendations:
- **Labeled Faces in the Wild (LFW)**: Download from http://vis-www.cs.umass.edu/lfw/
- **Custom Dataset**: Create folders with person names and add their images

### Installation Requirements:
```bash
pip install face_recognition opencv-python pillow numpy
pip install tensorflow keras-vggface keras-applications
```

---
## Approach 1: Using face_recognition Library (Easiest)
This uses a pre-trained ResNet model from dlib

In [2]:
# Import required libraries
import face_recognition
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
from pathlib import Path

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'face_recognition'

### Step 1: Create Dataset Structure
Create a folder structure like this:
```
dataset/
    person1/
        image1.jpg
        image2.jpg
    person2/
        image1.jpg
        image2.jpg
```

In [ ]:
# Configuration
DATASET_PATH = "dataset"  # Path to your dataset folder
TEST_IMAGE_PATH = "test_images"  # Path to test images

# Create directories if they don't exist
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(TEST_IMAGE_PATH, exist_ok=True)

print(f"Dataset folder: {DATASET_PATH}")
print(f"Test images folder: {TEST_IMAGE_PATH}")

### Step 2: Load and Encode Known Faces (Training)

In [ ]:
def load_known_faces(dataset_path):
    """
    Load all face images from the dataset and create encodings
    Returns: known_face_encodings, known_face_names
    """
    known_face_encodings = []
    known_face_names = []
    
    # Iterate through each person's folder
    for person_name in os.listdir(dataset_path):
        person_folder = os.path.join(dataset_path, person_name)
        
        # Skip if not a directory
        if not os.path.isdir(person_folder):
            continue
        
        print(f"Loading images for: {person_name}")
        
        # Load each image in the person's folder
        for image_name in os.listdir(person_folder):
            if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(person_folder, image_name)
                
                try:
                    # Load image and get face encoding
                    image = face_recognition.load_image_file(image_path)
                    encodings = face_recognition.face_encodings(image)
                    
                    if encodings:
                        # Take the first face encoding
                        known_face_encodings.append(encodings[0])
                        known_face_names.append(person_name)
                        print(f"  ✓ Loaded: {image_name}")
                    else:
                        print(f"  ✗ No face found in: {image_name}")
                except Exception as e:
                    print(f"  ✗ Error loading {image_name}: {str(e)}")
    
    print(f"\nTotal faces loaded: {len(known_face_encodings)}")
    return known_face_encodings, known_face_names

# Load the known faces
known_face_encodings, known_face_names = load_known_faces(DATASET_PATH)

if len(known_face_encodings) == 0:
    print("\n⚠️  No faces loaded! Please add images to the dataset folder.")
    print("   Create subfolders in 'dataset/' with person names and add their images.")

### Step 3: Face Recognition Function

In [ ]:
def recognize_faces(image_path, known_encodings, known_names, tolerance=0.6):
    """
    Recognize faces in an image
    
    Args:
        image_path: Path to the test image
        known_encodings: List of known face encodings
        known_names: List of names corresponding to encodings
        tolerance: How much distance between faces to consider a match (lower = more strict)
    
    Returns:
        Image with labeled faces, list of recognized names
    """
    # Load the test image
    image = face_recognition.load_image_file(image_path)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Find all face locations and encodings in the test image
    face_locations = face_recognition.face_locations(rgb_image)
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)
    
    recognized_names = []
    
    # Loop through each face found in the test image
    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # Compare face with known faces
        matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=tolerance)
        name = "Unknown"
        confidence = "?"
        
        # Calculate face distances
        face_distances = face_recognition.face_distance(known_encodings, face_encoding)
        
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            
            if matches[best_match_index]:
                name = known_names[best_match_index]
                confidence = f"{(1 - face_distances[best_match_index]) * 100:.1f}%"
        
        recognized_names.append(name)
        
        # Draw rectangle around face
        cv2.rectangle(rgb_image, (left, top), (right, bottom), (0, 255, 0), 2)
        
        # Draw label with name and confidence
        label = f"{name} ({confidence})"
        cv2.rectangle(rgb_image, (left, bottom - 35), (right, bottom), (0, 255, 0), cv2.FILLED)
        cv2.putText(rgb_image, label, (left + 6, bottom - 6), 
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    
    return rgb_image, recognized_names

print("Recognition function defined successfully!")

### Step 4: Test Face Recognition

In [ ]:
# Test with a single image
test_image = "test_images/test1.jpg"  # Change this to your test image path

if os.path.exists(test_image) and len(known_face_encodings) > 0:
    result_image, names = recognize_faces(test_image, known_face_encodings, known_face_names)
    
    # Display result
    plt.figure(figsize=(12, 8))
    plt.imshow(result_image)
    plt.axis('off')
    plt.title(f"Recognized: {', '.join(set(names))}")
    plt.show()
    
    print(f"Found {len(names)} face(s): {names}")
else:
    print("⚠️  Test image not found or no known faces loaded!")
    print(f"   Please add a test image at: {test_image}")

### Step 5: Batch Testing on Multiple Images

In [ ]:
def test_all_images(test_folder, known_encodings, known_names):
    """
    Test face recognition on all images in a folder
    """
    if not os.path.exists(test_folder):
        print(f"Test folder '{test_folder}' not found!")
        return
    
    test_images = [f for f in os.listdir(test_folder) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if not test_images:
        print(f"No images found in '{test_folder}'")
        return
    
    # Calculate grid size
    num_images = len(test_images)
    cols = min(3, num_images)
    rows = (num_images + cols - 1) // cols
    
    plt.figure(figsize=(15, 5 * rows))
    
    for idx, image_name in enumerate(test_images):
        image_path = os.path.join(test_folder, image_name)
        
        try:
            result_image, names = recognize_faces(image_path, known_encodings, known_names)
            
            plt.subplot(rows, cols, idx + 1)
            plt.imshow(result_image)
            plt.axis('off')
            plt.title(f"{image_name}\n{', '.join(set(names))}")
        except Exception as e:
            print(f"Error processing {image_name}: {str(e)}")
    
    plt.tight_layout()
    plt.show()

# Test on all images in test folder
if len(known_face_encodings) > 0:
    test_all_images(TEST_IMAGE_PATH, known_face_encodings, known_face_names)
else:
    print("⚠️  No known faces loaded. Please add images to the dataset folder first.")

### Step 6: Performance Evaluation (if you have labeled test data)

In [ ]:
def evaluate_accuracy(test_folder, known_encodings, known_names):
    """
    Evaluate accuracy if test images are organized in folders by person name
    Structure: test_images/person_name/image.jpg
    """
    correct = 0
    total = 0
    
    for person_name in os.listdir(test_folder):
        person_folder = os.path.join(test_folder, person_name)
        
        if not os.path.isdir(person_folder):
            continue
        
        for image_name in os.listdir(person_folder):
            if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(person_folder, image_name)
                
                try:
                    _, names = recognize_faces(image_path, known_encodings, known_names)
                    
                    if names and person_name in names:
                        correct += 1
                    total += 1
                except Exception as e:
                    print(f"Error: {e}")
    
    if total > 0:
        accuracy = (correct / total) * 100
        print(f"\nAccuracy: {accuracy:.2f}%")
        print(f"Correct: {correct}/{total}")
    else:
        print("No labeled test data found for evaluation.")

# Uncomment to evaluate (requires labeled test data)
# evaluate_accuracy(TEST_IMAGE_PATH, known_face_encodings, known_face_names)

---
## Approach 2: Using VGG-Face Pre-trained Model
This approach uses the VGG-Face model with TensorFlow/Keras

In [ ]:
# Import additional libraries for VGG-Face
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from keras_vggface.vggface import VGGFace
from keras_vggface.utils import preprocess_input
from tensorflow.keras.preprocessing import image
from scipy.spatial.distance import cosine

print(f"TensorFlow version: {tf.__version__}")
print("VGG-Face libraries imported successfully!")

### Load VGG-Face Pre-trained Model

In [ ]:
# Load VGG-Face model (this will download the model on first run)
# Using ResNet50 architecture
vgg_model = VGGFace(model='resnet50', include_top=False, input_shape=(224, 224, 3), pooling='avg')

print("VGG-Face model loaded successfully!")
print(f"Model input shape: {vgg_model.input_shape}")
print(f"Model output shape: {vgg_model.output_shape}")

### Extract Face Embeddings with VGG-Face

In [ ]:
def extract_face_vgg(image_path, required_size=(224, 224)):
    """
    Extract and preprocess face from image for VGG-Face
    """
    # Load image
    img = image.load_img(image_path, target_size=required_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array, version=2)  # version=2 for ResNet50
    return img_array

def get_face_embedding(model, face_pixels):
    """
    Get face embedding using VGG-Face model
    """
    embedding = model.predict(face_pixels, verbose=0)
    return embedding[0]

def load_faces_vgg(dataset_path, model):
    """
    Load all faces and create embeddings using VGG-Face
    """
    face_embeddings = []
    face_names = []
    
    for person_name in os.listdir(dataset_path):
        person_folder = os.path.join(dataset_path, person_name)
        
        if not os.path.isdir(person_folder):
            continue
        
        print(f"Processing: {person_name}")
        
        for image_name in os.listdir(person_folder):
            if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(person_folder, image_name)
                
                try:
                    face_pixels = extract_face_vgg(image_path)
                    embedding = get_face_embedding(model, face_pixels)
                    face_embeddings.append(embedding)
                    face_names.append(person_name)
                    print(f"  ✓ Processed: {image_name}")
                except Exception as e:
                    print(f"  ✗ Error: {image_name} - {str(e)}")
    
    print(f"\nTotal embeddings created: {len(face_embeddings)}")
    return np.array(face_embeddings), face_names

# Load faces with VGG-Face
if len(known_face_encodings) > 0:  # Only if dataset exists
    vgg_embeddings, vgg_names = load_faces_vgg(DATASET_PATH, vgg_model)
else:
    print("⚠️  Add images to dataset folder first!")

### Face Recognition with VGG-Face

In [ ]:
def recognize_face_vgg(test_image_path, model, known_embeddings, known_names, threshold=0.5):
    """
    Recognize face using VGG-Face embeddings
    
    Args:
        threshold: Cosine distance threshold (lower = more strict)
    """
    # Get embedding for test image
    test_pixels = extract_face_vgg(test_image_path)
    test_embedding = get_face_embedding(model, test_pixels)
    
    # Calculate distances to all known faces
    distances = []
    for known_embedding in known_embeddings:
        distance = cosine(test_embedding, known_embedding)
        distances.append(distance)
    
    # Find best match
    min_distance = min(distances)
    min_index = distances.index(min_distance)
    
    if min_distance <= threshold:
        name = known_names[min_index]
        confidence = (1 - min_distance) * 100
    else:
        name = "Unknown"
        confidence = 0
    
    return name, confidence, min_distance

# Test VGG-Face recognition
if os.path.exists(test_image) and 'vgg_embeddings' in locals():
    name, conf, dist = recognize_face_vgg(test_image, vgg_model, vgg_embeddings, vgg_names)
    
    # Display result
    img = plt.imread(test_image)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"VGG-Face Recognition\nName: {name}\nConfidence: {conf:.1f}%\nDistance: {dist:.3f}")
    plt.show()
    
    print(f"Predicted: {name} (confidence: {conf:.1f}%, distance: {dist:.3f})")
else:
    print("⚠️  Test image not found or embeddings not created!")

---
## Comparison and Analysis

In [ ]:
print("="*60)
print("FACE RECOGNITION COMPARISON")
print("="*60)
print("\nApproach 1: face_recognition (dlib ResNet)")
print("  Pros: Easy to use, good accuracy, handles multiple faces")
print("  Cons: Slower on large datasets")
print("  Best for: Quick prototyping, multiple faces in image")

print("\nApproach 2: VGG-Face (Deep CNN)")
print("  Pros: State-of-art accuracy, customizable, fast inference")
print("  Cons: Requires preprocessing, single face per image")
print("  Best for: Production systems, large-scale deployment")

print("\n" + "="*60)
print("RECOMMENDATION FOR YOUR PROJECT")
print("="*60)
print("✓ Use Approach 1 (face_recognition) for easier implementation")
print("✓ Use Approach 2 (VGG-Face) for better accuracy and control")
print("\nDataset: Start with LFW or create custom dataset with 5-10 images per person")

---
## Download LFW Dataset (Optional)

In [ ]:
# Uncomment to download LFW dataset
# from sklearn.datasets import fetch_lfw_people
# 
# # Download LFW dataset
# lfw_people = fetch_lfw_people(min_faces_per_person=20, resize=0.4)
# 
# print(f"Total dataset size: {lfw_people.images.shape[0]}")
# print(f"Number of classes: {len(lfw_people.target_names)}")
# print(f"Image shape: {lfw_people.images.shape[1:]}")
# print(f"\nPeople: {lfw_people.target_names}")

---
## Save Model/Embeddings (Optional)

In [ ]:
import pickle

# Save encodings for future use
def save_encodings(encodings, names, filename='face_encodings.pkl'):
    """
    Save face encodings and names to file
    """
    data = {'encodings': encodings, 'names': names}
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f"Encodings saved to {filename}")

def load_encodings(filename='face_encodings.pkl'):
    """
    Load face encodings and names from file
    """
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    print(f"Encodings loaded from {filename}")
    return data['encodings'], data['names']

# Example usage:
# save_encodings(known_face_encodings, known_face_names)
# known_face_encodings, known_face_names = load_encodings()

---
## Summary and Next Steps

### What you've learned:
1. ✓ Using pre-trained CNN models for face recognition
2. ✓ Two different approaches (face_recognition and VGG-Face)
3. ✓ Creating face embeddings/encodings
4. ✓ Testing and evaluating models

### Next steps for your project:
1. **Create your dataset**: Add 5-10 images per person in the dataset folder
2. **Test thoroughly**: Try different people and lighting conditions
3. **Tune parameters**: Adjust tolerance/threshold for better accuracy
4. **Add features**: 
   - Real-time webcam recognition
   - Face detection with anti-spoofing
   - Database integration
   - Attendance system

### Recommended datasets:
- **LFW**: http://vis-www.cs.umass.edu/lfw/ (13K images, 6K people)
- **CelebA**: https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html (200K images)
- **UTKFace**: https://susanqq.github.io/UTKFace/ (20K images with age/gender)
- **Yale Faces**: http://vision.ucsd.edu/content/yale-face-database (Good for small projects)

Good luck with your project! 🚀